In [1]:
using Pkg
using Plots
using ViscousStreaming
using HDF5
using JLD2

In [3]:
 Re = 40
 ϵ = 0.1
 Ω = 1.0 # frequency (keep this equal to 1)
 Tp = 2π/Ω # one period of oscillation
 p = StreamingParams(ϵ,Re)
 #s = StreamingAnalytical(p)

 τ = 0.1 # Stokes number, should be small
 β = 0.95 # Density parameter. Less than 1 means heavier than fluid.
 p_inert = InertialParameters(tau=τ,beta=β,epsilon=ϵ,Re=Re)
 #Ω = 1.0
 #Tp = 2π/Ω
 Tmax = 50*Tp

 Δx = 0.02
 xlim = (-3.8,3.8)
 ylim = (-3.8,3.8)
 n = 75
 body = Circle(0.2,n)

Circular body with 72 points and radius 0.2
   Current position: (0.0,0.0)
   Current angle (rad): 0.0


In [4]:
 bl = BodyList()
 bL1 = deepcopy(body)
 bL2 = deepcopy(body)
 bR1 = deepcopy(body)
 bR2 = deepcopy(body)

Circular body with 72 points and radius 0.2
   Current position: (0.0,0.0)
   Current angle (rad): 0.0


In [5]:

#Square
# left 1 cylinder
cent = (-2.0,2.0)
 α = 0.0
TL1 = RigidTransform(cent,α)
TL1(bL1) # transform the body to the current configuration

# left 2 cylinder
cent = (-2.0,-2.0)
 α = 0.0
TL2 = RigidTransform(cent,α)
TL2(bL2) # transform the body to the current configuration

# right 1 cylinder
cent = (2.0,2.0)
 α = 0.0
TR1 = RigidTransform(cent,α)
TR1(bR1) # transform the body to the current configuration

# right 2 cylinder
cent = (2.0,-2.0)
 α = 0.0
TR2 = RigidTransform(cent,α)
TR2(bR2) # transform the body to the current configuration

push!(bl,bL1);
push!(bl,bL2);
push!(bl,bR1);
push!(bl,bR2);


In [6]:
@time solver2 = FrequencyStreaming(Re,ϵ,Δx,xlim,ylim,bl);

 38.743506 seconds (1.23 G allocations: 54.757 GiB, 10.40% gc time)


In [7]:
ampvec = [ComplexF64[0.0,0.0] for i in 1:length(bl)];
 ampvec[1] = [1,0];  a2 = deepcopy(ampvec);
 ampvec[2] = [1,0];  a3 = deepcopy(ampvec); 
 ampvec[3] = [1,0];  a4 = deepcopy(ampvec); 
 ampvec[4] = [1,0];  a5 = deepcopy(ampvec);
 ampvec[1] = [0,0];  a6 = deepcopy(ampvec);
 ampvec[2] = [0,0];  a7 = deepcopy(ampvec);
 ampvec[3] = [0,0];  a8 = deepcopy(ampvec);
 ampvec[2] = [1,0];  a9 = deepcopy(ampvec);
 ampvec[1] = [1,0];  a10 = deepcopy(ampvec);
 ampvec[2] = [0,0];  a11 = deepcopy(ampvec);
 ampvec[3] = [1,0];  a12 = deepcopy(ampvec);
 ampvec[4] = [0,0];  a13 = deepcopy(ampvec);
 ampvec[1] = [0,0];  a14 = deepcopy(ampvec);
 ampvec[2] = [1,0];  a15 = deepcopy(ampvec);
 ampvec[3] = [0,0];  a16 = deepcopy(ampvec);

 actionspace = [a2, a3, a4, a5, a6, a7, a8, a9, a10, a11, a12, a13, a14, a15, a16];


In [8]:
strdVuxy = []
strdVvxy = []
for i = 1: length(actionspace)
    soln = solver2(actionspace[i],bl);
    isoln = inertial_velocity(soln,p_inert);
    v̄L = lagrangian_mean_velocity(isoln);
    v̄Luxy, v̄Lvxy = interpolatable_field(v̄L,isoln.g);
    push!(strdVuxy,v̄Luxy)
    push!(strdVvxy,v̄Lvxy)
end

In [ ]:
save("strdVuxy_square_bd2.jld2", "data", strdVuxy)
save("strdVvxy_square_bd2.jld2", "data", strdVvxy)

In [9]:
soln = solver2([ComplexF64[0.0,0.0] for i in 1:length(bl)],bl);

In [ ]:
plot(streamfunction(0,soln.s1),soln.g,levels=range(-1,1,length=31),xlim=xlim,ylim=ylim)
plot!(bl, layout = 1)

In [10]:
function mean_motion(dR,R,p,t,v̄Luxy,v̄Lvxy)
    dR[1] = v̄Luxy(R[1],R[2])
    dR[2] = v̄Lvxy(R[1],R[2])
   return dR 
end

mean_motion (generic function with 1 method)

In [11]:
function particle_trajectory_sample(positions::Vector{Tuple{Float64,Float64}}, n_particles)
    done = false

    @assert length(positions) == n_particles

    part_posx = [Vector{Vector{Float64}}() for _ in 1:n_particles]
    part_posy = [Vector{Vector{Float64}}() for _ in 1:n_particles]

    multiplier = 8500

    cyl_act = rand([7])
    
    v̄Lfcn1(dR,R,p,t) = mean_motion(dR,R,p,t,
            strdVuxy[cyl_act], strdVvxy[cyl_act])
    
    Tmax = multiplier * Tp

    for i in 1:n_particles

        sol = compute_trajectory(
            v̄Lfcn1,
            positions[i],
            Tmax,
            10Tp,
            bl=bl,
            ϵ=p.ϵ
        )

        push!(part_posx[i], sol[1,:])
        push!(part_posy[i], sol[2,:])
    end
    return part_posx, part_posy
end

particle_trajectory_sample (generic function with 1 method)

In [12]:
pos1, pos2 = particle_trajectory_sample([(0.0,0.0)], 1)
pos3, pos4 = particle_trajectory_sample([(1.1,1.1)], 1)
pos5, pos6 = particle_trajectory_sample([(1.2,-2.35)], 1)


([[[1.2, 1.293574858099169, 1.293574858099169, 1.4235751655748494, 1.4235751655748494, 1.5918226711044055, 1.5918226711044055, 1.739355946863697, 1.739355946863697, 1.7792496984682593  …  1.5459020302046786, 1.5459020302046786, 1.5452667691949555, 1.5452667691949555, 1.543447220949925, 1.543447220949925, 1.5408912009196796, 1.5408912009196796, 1.5381910652414623, 1.5381910652414623]]], [[[-2.35, -2.2855182878137072, -2.2855182878137072, -2.2250196889641645, -2.2250196889641645, -2.2037907789090703, -2.2037907789090703, -2.308593064731436, -2.308593064731436, -2.477302590529953  …  -2.4551857965343196, -2.4551857965343196, -2.4579961675417303, -2.4579961675417303, -2.4604897299521236, -2.4604897299521236, -2.4621078558926404, -2.4621078558926404, -2.4625137620406163, -2.4625137620406163]]])

In [13]:
#soln = solver2([ComplexF64[0.0,0.0] for i in 1:length(bl)], bl);
soln_temp = solver2(actionspace[7], bl);
xg, yg = coordinates(soln_temp.s1.W,solver2.grid)
plot(pos1, pos2, ratio=1,legend=false,linewidth=1,xlim=xlim,ylim=ylim,    
    tickfont = font(10, "Times"),
    size = (400, 400),
    markerstrokewidth = 0 
)
scatter!([pos1[end][end][end]], [pos2[end][end][end]])
plot!(pos3, pos4, ratio=1,legend=false,linewidth=1,xlim=xlim,ylim=ylim)
scatter!([pos3[end][end][end]], [pos4[end][end][end]])
plot!(pos5, pos6, ratio=1,legend=false,linewidth=1,xlim=xlim,ylim=ylim)
scatter!([pos5[end][end][end]], [pos6[end][end][end]])
#plot_1 = plot!(pos7, pos8, ratio=1,legend=false,linewidth=1,xlim=xlim,ylim=ylim)
plot!(lagrangian_mean_streamfunction(soln_temp), soln_temp.g, levels=15, color=:black, linewidth=0.5)
plot_1 = plot!(bl, fillcolor=:gray, linecolor=:gray)
#savefig(plot_1, "trajectories_sample.png")